In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import json
from datasets import load_from_disk
from inversion import *
import sys
sys.path.append('../white-box-inversion')
from transformers import AutoTokenizer
from utils import Eval
device='cuda'
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct')
evaluator = Eval(tokenizer=tokenizer, device=device)

def evalres(references_str, predictions_str, printmode='latex'):

    references_ids = [tokenizer.encode(x, add_special_tokens=False) for x in references_str]

    predictions_ids = [tokenizer.encode(x, add_special_tokens=False) for x in predictions_str]
    v = evaluator._text_comparison_metrics(predictions_ids, predictions_str, references_ids, references_str)
    if printmode == 'latex':
        print(f"{v['bge_emb_cos_sim_mean'] * 100:.2f}$\\pm$ {v['bge_emb_cos_sim_sem'] *100 :.2f} & {v['bleu_score']:.2f}$\\pm$ {v['bleu_score_sem']:.2f} & {v['rouge1_score']:.2f} & {v['exact_match'] * 100:.2f}$\\pm$ {v['exact_match_sem']:.2f} & {v['token_set_f1']*100:.2f} $\\pm$ {v['token_set_f1_sem']*100:.2f} ")
    else:
        print(f"{v['bge_emb_cos_sim_mean'] * 100:.2f}$\\pm$ {v['bge_emb_cos_sim_sem'] *100 :.2f} \t {v['bleu_score']:.2f}$\\pm$ {v['bleu_score_sem']:.2f} \t {v['rouge1_score']:.2f} \t {v['exact_match'] * 100:.2f}$\\pm$ {v['exact_match_sem']:.2f} \t {v['token_set_f1']*100:.2f} $\\pm$ {v['token_set_f1_sem']*100:.2f} ")
    return v

test_ds_mh = load_from_disk('../../data/black-box-inversion/long-context/inversion_mentalhealth_t5-base_l16_Meta-Llama-3.1-8B-Instruct')
invert_mh = json.load(open('../results/black-box/eval/mentalhealth-l3.18binstruct-l16-msl256-ep10.json', 'r'))
test_ds_ec = load_from_disk('../../data/black-box-inversion/long-context/inversion_evolcode_t5-base_l16_Meta-Llama-3.1-8B-Instruct')
invert_ec = json.load(open('../results/black-box/eval/evolcode-l3.18binstruct-l16-msl256-ep10.json', 'r'))

print('Results on Mentalhealth')
mh_eval = evalres(test_ds_mh['texts'], invert_mh['inference'])

print('Results on Evolcode')
ec_eval = evalres(test_ds_ec['texts'], invert_ec['inference'])

Results on Mentalhealth
97.33$\pm$ 0.25 & 70.57$\pm$ 2.18 & 0.85 & 0.00$\pm$ 0.00 & 88.77 $\pm$ 0.83 
Results on Evolcode
93.31$\pm$ 0.49 & 3.53$\pm$ 0.28 & 0.36 & 0.00$\pm$ 0.00 & 56.23 $\pm$ 1.24 
